In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns',50)

df_imp=pd.read_csv('Import.csv',sep=";",index_col=False,skip_blank_lines=True,quotechar='"',doublequote=True,on_bad_lines="skip",decimal=',',float_precision='round_trip')
df_exp=pd.read_csv('Export.csv',sep=";",index_col=False,skip_blank_lines=True,quotechar='"',doublequote=True,on_bad_lines="skip",decimal=',',float_precision='round_trip')

Two things i'll need to do:

1) Check if i have the data in the 2022<=>2025 timeframe as mentioned in the document
2) check enough info i need to know about my dataset (df.info())

In [2]:
print("\n############# Check Years ###############\n")

def check_years(df:pd.Series):
    df_to_list_sorted=df.unique().tolist()
    df_to_list_sorted.sort()
    return df_to_list_sorted

print(check_years(df_exp['Année']))
print(check_years(df_imp['Année']))

print("\n############# Info #############\n")

display(df_exp.info())
display(df_imp.info())


############# Check Years ###############

[2022, 2023, 2024, 2025]
[2022, 2023, 2024, 2025]

############# Info #############

<class 'pandas.DataFrame'>
RangeIndex: 12821 entries, 0 to 12820
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Code du chapitre SH                       12821 non-null  int64  
 1   Libellé du chapitre SH                    12821 non-null  str    
 2   Code du produit SH                        12821 non-null  int64  
 3   Libellé du produit SH                     12821 non-null  str    
 4   Code du groupement d'utilisation          12821 non-null  int64  
 5   Libellé du groupement d'utilisation       12821 non-null  str    
 6   Code du nouveau produits remarquables     12821 non-null  int64  
 7   Libellé du nouveau produits remarquables  12821 non-null  str    
 8   Code de la section CTCI                   12821 non-null  

None

<class 'pandas.DataFrame'>
RangeIndex: 64428 entries, 0 to 64427
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Code du chapitre SH                       64428 non-null  int64  
 1   Libellé du chapitre SH                    64428 non-null  str    
 2   Code du produit SH                        64428 non-null  int64  
 3   Libellé du produit SH                     64428 non-null  str    
 4   Code du groupement d'utilisation          64428 non-null  int64  
 5   Libellé du groupement d'utilisation       64428 non-null  str    
 6   Code du nouveau produits remarquables     64428 non-null  int64  
 7   Libellé du nouveau produits remarquables  64428 non-null  str    
 8   Code de la section CTCI                   64428 non-null  int64  
 9   Libellé de la section CTCI                64428 non-null  str    
 10  Code de la division CTCI                  644

None

For Import dataframe:
    * 3 columns have 1 null each, both have the type str

For Export dataframe:
    * 1 columns have 1 null each, it has the type str

we need to do the following:

1) drop the columns that have no analysis significance
2) Check cardinality

In [3]:
print('################ drop unuseful columns')

## are these columns really unuseful or would it help with the drillthrough ? 

cols_drop_exp=[col for col in df_exp.columns if 'code' in col.lower() and 'produit sh' not in col.lower() ]
cols_drop_imp=[col for col in df_imp.columns if 'code' in col.lower() and 'produit sh' not in col.lower() ]

df_exp=df_exp.drop(cols_drop_exp,axis=1)
df_imp=df_imp.drop(cols_drop_imp,axis=1)
display(df_exp.columns,df_imp.columns)

################ drop unuseful columns


Index(['Libellé du chapitre SH', 'Code du produit SH', 'Libellé du produit SH',
       'Libellé du groupement d'utilisation',
       'Libellé du nouveau produits remarquables',
       'Libellé de la section CTCI', 'Libellé de la division CTCI',
       'Libellé du groupe CTCI', 'Continent', 'Libellé du pays',
       'Libellé du flux', 'Année', 'Mois', 'Poids en KG', 'Valeur DHS',
       'Unité complementaire'],
      dtype='str')

Index(['Libellé du chapitre SH', 'Code du produit SH', 'Libellé du produit SH',
       'Libellé du groupement d'utilisation',
       'Libellé du nouveau produits remarquables',
       'Libellé de la section CTCI', 'Libellé de la division CTCI',
       'Libellé du groupe CTCI', 'Continent', 'Libellé du pays',
       'Libellé du flux', 'Année', 'Mois', 'Poids en KG', 'Valeur DHS',
       'Unité complementaire'],
      dtype='str')

In [4]:
print('################ Checking Cardinality')

def cardinality_check(df_name:str,df:pd.DataFrame):
    print(f"Checking Cardinality on {df_name}:\n")
    for col in df.columns:
        print(f"{col:<50}: Cardinality of {len(df[col].unique()):<10} data type : {df[col].dtype}")
    print(f'\n{30*"#"} Finished {30*"#"}\n')

## i forgot to change the dtypes before for code du produit
df_imp['Code du produit SH']=df_imp['Code du produit SH'].astype('str')
df_exp['Code du produit SH']=df_exp['Code du produit SH'].astype('str')

cardinality_check('export',df_exp)
cardinality_check('import',df_imp)

################ Checking Cardinality
Checking Cardinality on export:

Libellé du chapitre SH                            : Cardinality of 1          data type : str
Code du produit SH                                : Cardinality of 237        data type : str
Libellé du produit SH                             : Cardinality of 236        data type : str
Libellé du groupement d'utilisation               : Cardinality of 3          data type : str
Libellé du nouveau produits remarquables          : Cardinality of 9          data type : str
Libellé de la section CTCI                        : Cardinality of 2          data type : str
Libellé de la division CTCI                       : Cardinality of 4          data type : str
Libellé du groupe CTCI                            : Cardinality of 9          data type : str
Continent                                         : Cardinality of 7          data type : str
Libellé du pays                                   : Cardinality of 143        data 

i'll need to groupby libellé du produit sh & order by the count of elements inside the groupby

In [ ]:
## since the code column is higher than the libéllé column, i'll group them to see if it's really a typo or a duplicate.
df_exp_check=df_exp.groupby(by='Libellé du produit SH',group_keys=True)['Code du produit SH'].nunique()
df_imp_check=df_imp.groupby(by='Libellé du produit SH',group_keys=True)['Code du produit SH'].nunique()

lookup_val_exp=df_exp_check[df_exp_check>1].keys()
lookup_val_imp=df_imp_check[df_imp_check>1].keys()

def find_occurences(df_keys:pd.Index,df:pd.DataFrame,df_name:str):
    print(f"dataframe: {df_name}")
    col_name=df_keys.name
    pd.set_option("display.max_colwidth",None)
    for val in df_keys:
        df_check=df[df[col_name]==val][["Code du produit SH","Libellé du produit SH"]]
        display(df_check.drop_duplicates(ignore_index=True))
    pd.reset_option("display.max_colwidth")
# we then check which labels have more than one code

find_occurences(lookup_val_exp,df_exp,"Export")
find_occurences(lookup_val_imp,df_imp,"Import")

# x=df_exp_check[df_exp_check>1]
# y=df_imp_check[df_imp_check>1]
# print(x,y,sep="\n\n########\n\n")

### i'll need to check the full line to see if the year

dataframe: Export


,Code du produit SH,Libellé du produit SH
0,8703238600,"AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3"
1,8703238400,"AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3"


dataframe: Import


,Code du produit SH,Libellé du produit SH
0,8714100090,AUTRES PARTIES ET ACCESSOIRES DE MOTOCYCLES
1,8714109019,AUTRES PARTIES ET ACCESSOIRES DE MOTOCYCLES


,Code du produit SH,Libellé du produit SH
0,8701934091,"AUTRES TRACTEURS NEUFS, PUISSANCE ENTRE 37 KW ET 75KW"
1,8701939010,"AUTRES TRACTEURS NEUFS, PUISSANCE ENTRE 37 KW ET 75KW"


...

,Code du produit SH,Libellé du produit SH
0,8703238400,"AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3"
1,8703238600,"AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3"


,Code du produit SH,Libellé du produit SH
0,8703218200,"AUTRES VEHICULES NEUFS,A MOTEUR A PISTON ALTERN.CYL<=1000CM3"
1,8703218300,"AUTRES VEHICULES NEUFS,A MOTEUR A PISTON ALTERN.CYL<=1000CM3"


,Code du produit SH,Libellé du produit SH
0,8716800091,BROUETTES
1,8716809200,BROUETTES


,Code du produit SH,Libellé du produit SH
0,8704429919,"CHASSIS AVEC CABINE PR TRANSPORT MARCHANDISES, MOT. A PISTON ALLUM. PAR COMPRESSION ET ELECTRIQUE, 5T<CHARGE MAX<=20T"
1,8704439010,"CHASSIS AVEC CABINE PR TRANSPORT MARCHANDISES, MOT. A PISTON ALLUM. PAR COMPRESSION ET ELECTRIQUE, 5T<CHARGE MAX<=20T"


,Code du produit SH,Libellé du produit SH
0,8703338880,"VOITURES USAGEES,DIESEL OU SEMI,CYLINDREE SUP A 2500 CM3"
1,8703338890,"VOITURES USAGEES,DIESEL OU SEMI,CYLINDREE SUP A 2500 CM3"


it's not a data entry error, but rather the very nature of global SH vs Localized SH. (6 global code vs +4 special localized code)
i'll keep them both i suppose; i don't think there's a cardinality fix here unless if i would need to summarize data, that would make the loss of useful data.

## Questions i would like to answer (Export/Import) :

### *For Export*:

1) how much products we exported, by count, by sum (poids & valeur dhs)
2) how much revenue we generated by each groupement d'utilisation
3) for each groupement d'utilisation, what are the top 5 countries we export the products under that groupement d'utilisation (count vs valeur dhs)
4) 

### *For Import*:

1) 


### *Questions for Both of them (checking correlation or something interesting about the data)*:

1) 

# LAST NOTE BEFORE EXPORTING TO POWER BI

union the tables since they have libéllé de flux in it ( import or export )